<a href="https://colab.research.google.com/github/komalsathvik/DeepLearning/blob/main/ANN/ANN_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# from google.colab import files

# uploaded = files.upload()

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Once the file is uploaded, you can read it into a pandas DataFrame. Replace `your_file_name.csv` with the actual name of your uploaded file.

In [ ]:
#firstly upload the file into content section of colab
df=pd.read_csv("/content/powerplant_data.csv")
print(df)

         AT      V       AP     RH      PE
0      8.34  40.77  1010.84  90.01  480.48
1     23.64  58.49  1011.40  74.20  445.75
2     29.74  56.90  1007.15  41.91  438.76
3     19.07  49.69  1007.22  76.79  453.09
4     11.80  40.66  1017.13  97.20  464.43
...     ...    ...      ...    ...     ...
9563  15.12  48.92  1011.80  72.93  462.59
9564  33.41  77.95  1010.30  59.72  432.90
9565  15.99  43.34  1014.20  78.66  465.96
9566  17.65  59.87  1018.58  94.65  450.93
9567  23.68  51.30  1011.86  71.24  451.67

[9568 rows x 5 columns]


In [ ]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [ ]:
df.isnull().sum()

,0
AT,0
V,0
AP,0
RH,0
PE,0


In [ ]:
x=df.drop("PE",axis=1)
y=df["PE"]

In [ ]:
#we need to convert to data into tensors as well
# and we also need to do train test split as well
#firstly we will split our data
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)


In [ ]:
#start converting data into tensors
import torch
import torch.nn as nn

x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
#y is not of numpy array so we need to write .values

type(y_train)
#.view function is used for defining shape (rows,cols) is been passed
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
 from torch.utils.data import TensorDataset,DataLoader
 train_data=TensorDataset(x_train_tensor,y_train_tensor)
 test_data=TensorDataset(x_test_tensor,y_test_tensor)


In [ ]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
test_loader=DataLoader(test_data,batch_size=32)

**building our ANN Model**

# **DEEPLEARNING**

In [ ]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()

    self.model=nn.Sequential(
        #1st hidden layer
        nn.Linear(x_train.shape[1],6),
        nn.ReLU(),

        #2nd hidden layer
        nn.Linear(6,6),
        nn.ReLU(),

        #Output layer

        nn.Linear(6,1),
    )
  def forward(self,x):
    return self.model(x)

In [ ]:
model=ANN()

#loss fnx,optmizer
import torch.optim as optim
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [47]:
#training the model
epochs=100
train_losses=[]
validation_losses=[]
best_validation_loss=float('inf') #initalized with infinity to get min loss

for epoch in range(epochs):
  model.train()
  running_loss=0.0
  #xb=features of one batch
  #yb=labels of one batch
  for xb,yb in train_loader:
      optimizer.zero_grad() #clearing the gradients
      outputs=model(xb)
      loss=criterion(outputs,yb)
      loss.backward() #back propagation
      optimizer.step() #params update
      running_loss+= loss.item() #loss is a tensor so converting into a py float
   #validation
  epoch_loss=running_loss/len(train_loader)
  train_losses.append(epoch_loss)
  if(epoch_loss<best_validation_loss):
    best_validation_loss=epoch_loss
    torch.save(model.state_dict(),"best_model.pth")
  with torch.no_grad():
      running_validation=0.0
      for xb,yb in test_loader:
        outputs=model(xb)
        loss=criterion(outputs,yb)
        running_validation+=loss.item()
  epoch_validation_loss=running_validation/len(test_loader)
  validation_losses.append(epoch_validation_loss)

print("Training Loss:",train_losses[-1])
print("Validation Loss:",validation_losses[-1])

Training Loss: 19.53995539744695
Validation Loss: 17.937553358078002


In [49]:
#loading the best model
#evaluation as well
torch.load("best_model.pth")
model.eval()

ANN(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=6, bias=True)
    (1): ReLU()
    (2): Linear(in_features=6, out_features=6, bias=True)
    (3): ReLU()
    (4): Linear(in_features=6, out_features=1, bias=True)
  )
)

In [50]:
with torch.no_grad():
  train_preds=model(x_train_tensor)
  test_preds=model(x_test_tensor)

  train_mse_loss=criterion(train_preds,y_train_tensor)
  test_mse_loss=criterion(test_preds,y_test_tensor)
print("Train MSE Loss:",train_mse_loss.item())
print("Test MSE Loss:",test_mse_loss.item())

Train MSE Loss: 19.294233322143555
Test MSE Loss: 17.941650390625


In [53]:
from sklearn.metrics import r2_score

print("r^2 score =" ,r2_score(y_test,test_preds))

r^2 score = 0.9372985592757028
